# Agentowy asystent RAG



### A. Co budujesz

Asystenta **agentowego**, nie klasycznego RAG-a. Klasyczny RAG to sztywno ustalony przebieg postępowania:

(`embed → pobierz → wklej kontekst → generuj`)

 — zawsze te same kroki, żadnej decyzji.

**Agent** sam decyduje,
czy, kiedy i którego narzędzia użyć, ogląda wynik i może wykonać kolejny krok.

### B. Proponowana zawężona tematyka PDF

Zbierz **3–5 PDF-ów na jeden, spójny temat** i wrzuć do folderu.

**Przepisy kulinarne** — trzeba nakarmić innych rozbitków. Przykładowe drugie narzędzie:
przeliczanie porcji, lista zakupów z kilku przepisów, porównanie z zawartością spiżarni, sumowanie czasu przygotowania.

**Instrukcje obsługi** (jakieś przedmioty przydatne do przetrwania). Przykładowe drugie narzędzie:
zużycie paliwa, przelicznik jednostek (PSI ↔ bar, °F ↔ °C), harmonogram serwisu, wyszukiwarka kodów błędów.

Drugie narzędzie ma mieć **weryfikowalne wyjście** — takie, po którym widać, że policzyło, a nie zgadło.
Narzędzie, które tylko przepisuje tekst albo którego agent nigdy nie wywoła, się nie liczy.

### C. Wybór narzędzi

| Warstwa | Czego najlepiej użyć |
|---|---
| Parsowanie PDF | PyMuPDF
| Podział na fragmenty | `RecursiveCharacterTextSplitter` |
| Embeddingi | model z HuggingFace |
| Baza wektorowa | Chroma lub Qdrant |
| Model generatywny | darmowe API z **natywnym tool callingiem** (Groq, Gemini) |
| Agent | `create_agent` z LangChain |
| Pamięć | `InMemorySaver` z LangGraph |

### D. Polecenie:

- Załaduj 5 dokumentów PDF dotyczących wybranej tematyki, podział na strony, umieść je w folderze na dysku
- Pamiętaj o kluczu do LLM, trzymaj go w folderze `.env`
- Podziel strony na chunki
- Wybierz model do embeddingów z HuggingFace
- Utwórz bazę wektorową i wypełnij ją danymi
- Zdefiniuj **co najmniej** 2 narzędzia
- Zbuduj agenta, który opiera się na ustalonym prompcie systemowym
- Sprawdź jak działa, zadając pytania (nie)dotyczące dokumentów z bazy

**Dodatkowe wymagania:**
- Mechanizm pamięci konwersacji - tak, żeby model pamiętał poprzednią część konwersacji
- Mechanizm braku halucynacji - jeśli model nie znajdzie odpowiedzi w bazie, informuje o tym
- Cytowanie dokumentu, z którego model pobrał informacje

**Dla chętnych:**
- Model-sędzia (LLM-as-judge) - ocena generowanych odpowiedzi (faithfulness, answer relevancy)
- Retrieval metrics: hit rate@k, MRR
- Rewriting pytań zależnych od kontekstu (zaimki) przed retrievalem
- Lepsze czyszczenie PDF (OCR, czyszczenie), tuning chunk_size/chunk_overlap

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza Markdown.

### 1. Środowisko i dane

In [1]:
!pip install python-dotenv pymupdf langchain langchain-google-genai langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers chromadb langgraph

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

**Uzasadnienie wyboru narzędzi**

- **Gemini zamiast Groq**: logowanie do Groq (przez GitHub/Google OAuth) nie
  działało niezależnie od przeglądarki i konta. Google AI Studio zadziałało bez
  problemu i również oferuje darmowy tier z natywnym tool callingiem.
- **Chroma zamiast Qdrant**: baza działa lokalnie w procesie, bez potrzeby
  stawiania osobnego serwera — wystarczające przy tak małej bazie. Qdrant miałby sens przy większej skali.
- **Model embeddingów**: `paraphrase-multilingual-MiniLM-L12-v2` zamiast
  standardowego angielskiego `all-MiniLM-L6-v2`, ponieważ dokumenty i
  zapytania są w języku polskim, a model multilingual lepiej radzi sobie
  z polską semantyką.

### 2. Pobieranie pdfów

In [3]:
import pymupdf
from pathlib import Path

# Funkcja do wczytywania danych z plików pdf
def load_pdfs_from_folder(folder_path: str) -> list[dict]:
    pages = []
    pdf_folder = Path(folder_path)

    for pdf_file in pdf_folder.glob("*.pdf"):
        doc = pymupdf.open(pdf_file)

        for page_num, page in enumerate(doc):
            text = page.get_text()

            if not text.strip():
                continue

            pages.append({
                "text": text,
                "source": pdf_file.name,
                "page": page_num + 1,
            })

    return pages

In [4]:
# Sprawdzenie czy pliki zostały poprawnie pobrane
all_pages = load_pdfs_from_folder("pdf")
print(f"Wczytano {len(all_pages)} stron z {len(set(p['source'] for p in all_pages))} plików")

Wczytano 10 stron z 5 plików


### 3. Chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Rozbicie plików na kawałki oraz zamiana na documenty
def split_into_chunks(pages: list[dict], chunk_size: int = 500, chunk_overlap: int = 50) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    documents = []
    for page in pages:
        chunks = splitter.split_text(page["text"])

        for chunk in chunks:
            doc = Document(
                page_content=chunk,
                metadata={"source": page["source"], "page": page["page"]},
            )
            documents.append(doc)

    return documents

In [6]:
# Sprawdzenie poprawności podziału chunków
chunks = split_into_chunks(all_pages)
print(f"Powstało {len(chunks)} chunków z {len(all_pages)} stron")

Powstało 22 chunków z 10 stron


### 4. Embedding

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### 5. Baza wektorowa


In [8]:
import shutil
import os
from langchain_chroma import Chroma

# Tworzenie bazy wektorowej oraz jej wypełnienie. Dodatkowo czyścimy przy ponownym odpaleniu by uniknąć duplikatów.
def build_vector_store(documents: list[Document], embedding_model, persist_directory: str = "chroma_db"):
    if os.path.exists(persist_directory):
        shutil.rmtree(persist_directory)

    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
    )

    return vector_store

In [9]:
# Sprawdzenie ilości powstałych wektorów
vector_store = build_vector_store(chunks, embedding_model)
print(vector_store._collection.count())

22


### 6.1 Narzędzie - szukanie przepisów

In [37]:
from langchain_core.tools import tool

@tool
def search_recipes(query: str) -> str:
    """
    Przeszukuje bazę przepisów kulinarnych i zwraca najbardziej pasujące fragmenty
    wraz z informacją o źródle (nazwa pliku i numer strony).
    Użyj tego narzędzia, gdy użytkownik pyta o konkretny przepis, składniki lub sposób przygotowania.
    """
    results = vector_store.similarity_search(query, k=5)

    if not results:
        return "Nie znaleziono żadnych pasujących przepisów w bazie."

    formatted = []
    for r in results:
        formatted.append(f"{r.page_content}\n[Źródło: {r.metadata['source']}, strona {r.metadata['page']}]")

    return "\n\n".join(formatted)

6.2 Narzędzie - liczenie proporcji

In [11]:
@tool
def przelicz_porcje(skladniki: list[str], ilosc_oryginalna: list[float], porcje_oryginalne: int, porcje_docelowe: int) -> str:
    """
    Przelicza ilość składników z przepisu na inną liczbę porcji.
    Użyj tego narzędzia, gdy użytkownik chce zmienić liczbę porcji przepisu
    (np. "przelicz na 6 porcji", "ile potrzebuję na 2 osoby zamiast 4").
    Listy powinny mieć podane składniki i ich ilość w tej samej kolejności.

    Args:
        skladniki: lista składników (np. "mąka", "jajka")
        ilosc_oryginalna: lista ilości składników w oryginalnym przepisie
        porcje_oryginalne: liczba porcji w oryginalnym przepisie
        porcje_docelowe: docelowa liczba porcji, na którą przeliczamy
    """
    if porcje_oryginalne <= 0:
        return "Błąd: liczba porcji oryginalnych musi być większa od zera."

    if porcje_docelowe <= 0:
        return "Błąd: liczba porcji docelowych musi być większa od zera."

    wyniki = []
    for i in range(len(skladniki)):
        nowa_ilosc = ilosc_oryginalna[i] * (porcje_docelowe / porcje_oryginalne)
        wyniki.append({
            "skladnik": skladniki[i],
            "ilosc_oryginalna": ilosc_oryginalna[i],
            "ilosc_nowa": nowa_ilosc,
        })

    text = ""
    for w in wyniki:
        text += f"{w['skladnik']}: {w['ilosc_oryginalna']} (na {porcje_oryginalne} porcji) → {w['ilosc_nowa']:.2f} (na {porcje_docelowe} porcji)\n"

    return text

### 7. Agent

In [33]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

# Wykorzystanie modelu googla
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

system_prompt = """Jesteś asystentem kulinarnym, który pomaga rozbitkom na bezludnej wyspie.
Odpowiadasz WYŁĄCZNIE na podstawie treści znalezionych w dostępnych dokumentach (przepisach).

Zasady:
1. Zawsze najpierw użyj narzędzia search_recipes, żeby sprawdzić czy odpowiedź jest w bazie.
2. Jeśli pytanie dotyczy przeliczenia porcji, użyj narzędzia przelicz_porcje na podstawie
   ilości znalezionych w bazie.
3. Jeśli nie znajdziesz odpowiedzi w dokumentach, wprost powiedz: "Nie mam tej informacji
   w dostępnych przepisach." NIE zgaduj i nie wymyślaj odpowiedzi.
4. Zawsze podawaj źródło (nazwa pliku, numer strony) informacji, której użyłeś w odpowiedzi.
5. Odpowiadaj w języku polskim, zwięźle i konkretnie.
"""

In [34]:
agent = create_agent(
    model=llm,
    tools=[search_recipes, przelicz_porcje],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

### 8. Testy

In [17]:
config = {"configurable": {"thread_id": "rozbitek-1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Jak zrobić rosół?"}]},
    config=config,
)
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Aby zrobić rosół drobiowy z makaronem, postępuj zgodnie z poniższym przepisem:\n\n### **Składniki (na 4 porcje):**\n* 1 kg skrzydełek i podudzi z kurczaka\n* 2 litry wody\n* 2 marchewki\n* 1 pietruszka (korzeń)\n* 1/4 selera\n* 1 por\n* 1 cebula\n* 2 liście laurowe\n* 4 ziarna ziela angielskiego\n* 1 łyżeczka soli\n* 1/2 łyżeczki pieprzu\n* 150 g makaronu nitki\n\n### **Sposób przygotowania:**\n1. Mięso zalej zimną wodą i doprowadź do wrzenia, regularnie zbierając szumowiny.\n2. Cebulę przekrój na pół i opal na suchej patelni.\n3. Dodaj warzywa (marchewki, pietruszkę, seler, por), opaloną cebulę oraz przyprawy (liście laurowe, ziele angielskie, sól, pieprz) do garnka.\n4. Gotuj całość na małym ogniu pod przykryciem przez 75 minut.\n5. Odcedź rosół, a makaron ugotuj osobno.\n6. Podawaj gorący rosół z makaronem i pokrojoną marchewką.\n\n*Źródło: 01_zupy.pdf, strona 1*', 'extras': {'signature': 'EoALCv0KARFNMg+gZaJypzjwzZ+hwqb6jnT0jkFXVqmVWJ/ZQP6qOn3MGtWCkrPYRT6

In [18]:
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Na ile porcji był ten przepis?"}]},
    config=config,
)
print(response2["messages"][-1].content)

[{'type': 'text', 'text': 'Ten przepis był przewidziany na **4 porcje**.\n\n*Źródło: 01_zupy.pdf, strona 1*', 'extras': {'signature': 'EvkCCvYCARFNMg+SXlbKlPX2YnDkOToNptzfpHbZIvsZ4N8AtG9XT8B0fLyairOaOq55S5Z9GDJ6aFIQNqz1BGyNziiwOKnwBOCVtByQ+5LdazQ+ywezgIQ/0fk+/4d9M+leTrHT+/rSfCa/4fVNRVNGJ4/rINhSDMBdfEpvvU14AE3it8tkIRV+a8/rPEeJCHwnKrm0GkBZs3RxO2Y92kfp52G3MTn0XoWGjKsdkBZWZI8dMTQ7X6J7x2bhgwvE1uXk3KdZ9NV3uALw02UG3MNbrw/sEQdUAGoIRgv9td6QxclWn1MjpOLgU0DC+oqljw9bg72l8SjgI0bwCf/aImjalfucYvrH49WxaIA0NP+vESdbhYFgwPef9617n1h8o2R/n6upGLvMBq1vmfoPGItXJ6hDLbSHgo87k4dRdnUyk0FJkgquhq4b8FNoSsOLFcC97VCDmHBoDMlT1+EHJPZsUEoO9HrrpdxIHSV/xR6V287H5bwLuOc0RAs='}}]


In [19]:
config2 = {"configurable": {"thread_id": "rozbitek-2"}}
response3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Jak zrobić sushi?"}]},
    config=config2,
)
print(response3["messages"][-1].content)

[{'type': 'text', 'text': 'Nie mam tej informacji w dostępnych przepisach.', 'extras': {'signature': 'EpwDCpkDARFNMg+JWE4vNFhfEHgqcOsZFJdTOmJ5XXi6AylddWuuRIZ/DQrxGCKWcUjqHfbwemRSvrWMY3qDEizinZ2ks/TTWMdmsfCYPJpsWiayWxocgHPKs0g9v+4QxjtPixXrEC1Do1ryumtqhNLTP+c2FcQ+F7Hq6ZdDzAM0NlKP/gdo+FUUQ042DG30SJQJj158bQP80zZ7CuiVc6JNFrH/7tDh0pwK9Rsr+uXeN9P5LCAXiBFwYvMCNAYImuZuUzgjV6B6JigRZA1rrWLHVOkPb3DgK9yzpMib5BMftWNknHWWAgHv/Af98f43PKPfti/OuJkAlp80y6DyR5sRxz2EGkIsx+lxAVrC+NST/g5t+Zk+Mt7Li39zi8RXX1Cnl4ojIAcTtWLyL6vn7r0h/lAJZWLHtZlQ7HVXIe2JLQypGrtzZSYHHd1anySQe8JH9ZtMKcFpdqVlxUIwub/lWDIq4Uv+WXykH3nIMwMyYWehQTrix7rrsYACEow//VMTIB6as1K1+YqUKKEMYmzie73QkEtjHqIIm717TA=='}}]


In [35]:
config3 = {"configurable": {"thread_id": "rozbitek-3"}}

response4 = agent.invoke(
    {"messages": [{"role": "user", "content": "Mam przepis na szarlotkę na 8 porcji. Ile składników potrzebuję na 3 porcje?"}]},
    config=config3,
)
print(response4["messages"][-1].content)

[{'type': 'text', 'text': 'Aby przygotować **Szarlotkę klasyczną** na 3 porcje (zamiast 8), potrzebujesz następujących ilości składników:\n\n* **Mąka pszenna:** 112,5 g\n* **Masło:** 75 g\n* **Cukier:** 37,5 g\n* **Jajko:** 0,38 szt. (ok. 3/8 jajka)\n* **Proszek do pieczenia:** 0,38 łyżeczki\n* **Jabłka:** 375 g\n* **Cynamon:** 0,38 łyżeczki\n* **Cukier do jabłek:** 0,75 łyżki\n\n[Źródło: 04_desery.pdf, strona 1]', 'extras': {'signature': 'EuAFCt0FARFNMg9I+Ctv+hGRK1mbBXGjgCsYR867S2fCWCEtjJDXPOrOCagyeA5X2gd7oq3jRh9ziU9ghdNfcAHt8E3RdufnWOj0730rT2Rzxo9K+Yrh8+iSYokdOhs5xFqmcH9/avPZ9Dr4c2vJppkBLrYYJmvNerJY4m3UIcmy2/tfcNSiwPJBb71+Gz8K689f0VPRyfb+oM9mYMcZwSOdDpgMeIZRn+Z24Ic9zwZlE/Kni1IbH1IdQ4xCBxnRyNSbAGtCY1BLygIJEvhNJheb+B9J5DpADWiN+K5c7Tc7KxYbxmTkrckaLzswoDcR41XanzFNPO/F04RdfQjjbSdbco8mpRwuCLHKfmvHMYcZ9MDHGK5ghJnhwkhQuOxDxSxJob9fZ3zCl6rk06yW7icISziG0F+BJ0p4K71FuNLxi+WWqDwOcC1/1T4DLbBNVRNI6qMMS2gXHHln//otca+azla/1tFcW9PuemTVMpd0D3Em16tlCNABL9vmSpvxmHx8iglI2ZjMMF+QMguMIZqaannrlk+FwFCsYomojgPE

In [36]:
config4 = {"configurable": {"thread_id": "rozbitek-4"}}

response5 = agent.invoke(
    {"messages": [{"role": "user", "content": "Jaka jest stolica Francji?"}]},
    config=config4,
)
print(response5["messages"][-1].content)

[{'type': 'text', 'text': 'Nie mam tej informacji w dostępnych przepisach.', 'extras': {'signature': 'EqcDCqQDARFNMg9GQsjsase/aW/yJ37/SXdbFy9JCWnhprhSFyXr9CNefof8QF9yZ9kUr1zZy8EYACxUrV1pB8GWKreFB96GARSnAHkNIXkx4qvUK47sCdpuD0XNVKpomr5R7qInkN0XTsfpsdg/a1EK0jxaZCquQXbqReVJStButMv2vy48I9TDYlS/KxmQIQaxjCAP1zR0BLcQd+NNxfADQDuFXCESw7El0v1JzqzOYMR/Q47J9aGblANeaSM9JZ6bFdrJF+fJ4SzO06c14d6MtLKhfWwFrGbtNox33Z1z7N+UI4yR/wuGRitf2Ge1SGz3cQR7CGijekTX41UcFJbUmrS+EgLysuZRYB72Ti/LhrLTSsdJdaODWZCrcusRICxE+spV1aAE/uyJ2CD2acpbej2VvjtlOdJGjq5ybtS2jIZ2CXoQdddy73EHZrJMswN35kH5c69J2d31/l0ruD2EeYxcRieLjFANZfLt8cqk2Shkd7Cn6KeM1wmM1Xd1uuThRernPxLqgZbESR581pt1pxxEygrXtjUW0vb2BrqqzWlUxHqamvxH'}}]


Generalnie to wszystkie działają tylko API calle się skończyły już 3 razy. Najwidoczniej jedno `.invoke()` robi więcej niż jedno bo z 3 wywiołań zrobiło się 13 calli. Dodatkowo jak testowałem modele `light` to radziły one sobie dużo gorzej od zwykłych i wymagały dużo bardziej dokładnych sformółowań.

PS. jednak jeden model się zresetował więc są wszystkie odpowiedzi.